# CHỌN MẪU FFHQ KHỚP PHÂN PHỐI FG-NET (CHO BÁO CÁO SO SÁNH 2 DATASET)

Notebook này chọn 82 người từ FFHQ-Aging (70.000 ảnh có nhãn), khớp đúng phân phối
nhóm tuổi nguồn với 82 người của FG-NET (batch v3 chính thức), để 2 bảng metrics
báo cáo (FG-NET vs FFHQ) so sánh công bằng — chỉ để domain ảnh cũ/mới là biến khác
nhau, các biến gây nhiễu khác (cỡ mẫu, phân phối tuổi, phân phối gap) được kiểm soát.

**QUAN TRỌNG**: Cell 3 cần file kết quả FG-NET **v3 THẬT** (164 cặp, aging-only,
LocalBlend + preprocessing) — không phải file kết quả cũ (230 cặp, có cả chiều
trẻ hóa, baseline 9.48% trước khi fix). Notebook tự kiểm tra và báo lỗi nếu phát
hiện file không khớp bản v3.

**Thứ tự chạy**: chạy batch v3 FG-NET (164 cặp) xong trước → chạy notebook này →
dùng `ffhq_selected_82_persons.csv` output để tạo cặp FFHQ trong notebook pipeline chính.

## 1. Cấu hình & Đường dẫn

In [ ]:
import pandas as pd
import numpy as np
import os

RANDOM_SEED = 42
N_PERSONS = 82
MAX_PAIRS_PER_PERSON = 2
N_DISTRACTORS = 200  # ảnh phụ làm nhiễu cho gallery FAISS, không phải đối tượng truy vấn

# ===== ĐƯỜNG DẪN DỮ LIỆU (KAGGLE) — sửa lại nếu cấu trúc dataset khác =====
FFHQ_LABELS_CANDIDATES = [
    "/kaggle/input/datasets/menonkk/nckh-2025-2026/ffhq_aging_labels.csv",
    "/kaggle/working/ffhq_aging_labels.csv",
    "d:/Data/project/nckh/data/ffhq_aging_labels.csv",
]
FGNET_V3_RESULTS_CANDIDATES = [
    "/kaggle/working/fgnet_eval_results.csv",  # file output thật của notebook v3 pipeline
    "d:/Data/project/nckh/data/fgnet_eval_results.csv",
]

FFHQ_LABELS_CSV = next((p for p in FFHQ_LABELS_CANDIDATES if os.path.isfile(p)), None)
FGNET_V3_RESULTS_CSV = next((p for p in FGNET_V3_RESULTS_CANDIDATES if os.path.isfile(p)), None)

assert FFHQ_LABELS_CSV is not None, "❌ Không tìm thấy ffhq_aging_labels.csv — kiểm tra lại đường dẫn."
print(f"✅ FFHQ labels: {FFHQ_LABELS_CSV}")
print(f"{'✅' if FGNET_V3_RESULTS_CSV else '⚠️ '} FG-NET v3 results: {FGNET_V3_RESULTS_CSV or 'CHƯA TÌM THẤY — sẽ dùng placeholder minh họa'}")

# ===== NGƯỠNG LỌC CHẤT LƯỢNG — khớp tinh thần checklist "Ảnh đầu vào tốt" đã dùng cho FG-NET =====
MIN_AGE_GROUP_CONFIDENCE = 0.8
MIN_GENDER_CONFIDENCE = 0.8
MAX_HEAD_ANGLE_DEG = 15.0
MAX_EYE_OCCLUSION = 0.3

## 2. Lọc chất lượng ảnh FFHQ

Loại bỏ ảnh có nhãn không chắc chắn (confidence thấp), góc nghiêng đầu lớn, hoặc mắt
bị che khuất — cùng tiêu chí đã dùng để đánh giá "ảnh đầu vào tốt" cho FG-NET, đảm
bảo 2 dataset được lọc ở cùng 1 mức chuẩn chất lượng trước khi đưa vào pipeline.

In [ ]:
def load_and_filter_ffhq(path: str) -> pd.DataFrame:
    df = pd.read_csv(path)
    before = len(df)
    df = df[
        (df["age_group_confidence"] >= MIN_AGE_GROUP_CONFIDENCE)
        & (df["gender_confidence"] >= MIN_GENDER_CONFIDENCE)
        & (df["head_yaw"].abs() <= MAX_HEAD_ANGLE_DEG)
        & (df["head_pitch"].abs() <= MAX_HEAD_ANGLE_DEG)
        & (df["left_eye_occluded"] < MAX_EYE_OCCLUSION)
        & (df["right_eye_occluded"] < MAX_EYE_OCCLUSION)
    ].copy()
    print(f"Lọc chất lượng: {before} -> {len(df)} ảnh còn lại ({len(df)/before*100:.1f}%)")
    return df


ffhq_df = load_and_filter_ffhq(FFHQ_LABELS_CSV)
print("\nPhân phối nhóm tuổi SAU khi lọc chất lượng (nguồn để chọn mẫu):")
print(ffhq_df["age_group"].value_counts().sort_index())

## 3. Trích phân phối mục tiêu từ kết quả FG-NET v3 THẬT

Đếm phân phối nhóm tuổi nguồn của 82 người FG-NET (quy đổi tuổi cụ thể sang đúng
10 bucket nhãn của FFHQ), và lấy toàn bộ 164 giá trị gap tuổi thật để dùng sample
gap cho FFHQ ở bước sau. Có 2 lớp kiểm tra an toàn để tránh lặp lại nhầm lẫn đã
từng gặp (dùng nhầm file kết quả cũ, 230 cặp, trộn cả chiều trẻ hóa).

In [ ]:
def age_to_group(age: float) -> str:
    """Quy đổi 1 tuổi cụ thể (FG-NET) sang đúng nhãn age_group của FFHQ."""
    bounds = [(0, 2), (3, 6), (7, 9), (10, 14), (15, 19),
              (20, 29), (30, 39), (40, 49), (50, 69), (70, 120)]
    labels = ["0-2", "3-6", "7-9", "10-14", "15-19",
              "20-29", "30-39", "40-49", "50-69", "70-120"]
    for (lo, hi), label in zip(bounds, labels):
        if lo <= age <= hi:
            return label
    raise ValueError(f"Tuổi {age} không khớp bucket nào")


def build_fgnet_target_profile(fgnet_csv: str):
    fg = pd.read_csv(fgnet_csv)

    # Cảnh báo an toàn — tự kiểm tra file có đúng là bản v3 không
    assert len(fg) <= 170, (
        f"File có {len(fg)} dòng — không giống batch v3 (~164 cặp). "
        "Kiểm tra lại có đang dùng nhầm file kết quả cũ không."
    )
    gaps = fg["target_age"] - fg["source_age"]
    assert (gaps >= 0).all(), (
        "Phát hiện gap âm (trẻ hóa) trong file — đây có thể là file cũ trộn cả 2 "
        "chiều, không phải batch v3 (chỉ giữ chiều già hóa)."
    )

    unique_persons = fg.drop_duplicates("person_id")
    age_groups = unique_persons["source_age"].apply(age_to_group)
    target_profile = age_groups.value_counts().to_dict()

    return target_profile, gaps.tolist()


if FGNET_V3_RESULTS_CSV is not None:
    target_profile, gap_distribution = build_fgnet_target_profile(FGNET_V3_RESULTS_CSV)
    print("✅ Đã trích phân phối THẬT từ FG-NET v3:")
else:
    print("⚠️  CHƯA CÓ file kết quả FG-NET v3 thật — dùng placeholder minh họa.")
    print("    Chạy batch v3 164 cặp TRƯỚC, rồi chạy lại notebook này với file thật.\n")
    # Placeholder CHỈ để minh họa cấu trúc — KHÔNG dùng số này cho báo cáo chính thức
    target_profile = {
        "0-2": 3, "3-6": 6, "7-9": 7, "10-14": 10, "15-19": 8,
        "20-29": 14, "30-39": 12, "40-49": 10, "50-69": 8, "70-120": 4,
    }
    gap_distribution = list(np.random.default_rng(0).integers(3, 40, size=164))

assert sum(target_profile.values()) == N_PERSONS, (
    f"Tổng phân phối ({sum(target_profile.values())}) khác N_PERSONS ({N_PERSONS}) "
    "— kiểm tra lại logic trích xuất từ FG-NET."
)

for k, v in sorted(target_profile.items()):
    print(f"  {k:>8}: {v} người")

## 4. Chọn 82 người FFHQ khớp phân phối FG-NET

Với mỗi nhóm tuổi, chọn ngẫu nhiên (seed cố định) đúng số lượng cần từ pool FFHQ đã
lọc chất lượng ở bước 2 — không ép khớp giới tính (FG-NET không có cột gender trong
file kết quả hiện tại), chỉ in ra để tham khảo.

In [ ]:
def select_ffhq_samples(ffhq_df: pd.DataFrame, target_profile: dict, rng: np.random.Generator):
    selected_parts = []
    for age_group, n_needed in target_profile.items():
        pool = ffhq_df[ffhq_df["age_group"] == age_group]
        if len(pool) < n_needed:
            raise ValueError(
                f"Nhóm {age_group}: cần {n_needed} ảnh nhưng chỉ có {len(pool)} "
                "ảnh đạt chất lượng — hạ ngưỡng lọc ở Cell 2 hoặc giảm N_PERSONS."
            )
        chosen_idx = rng.choice(pool.index, size=n_needed, replace=False)
        selected_parts.append(ffhq_df.loc[chosen_idx])
    return pd.concat(selected_parts).reset_index(drop=True)


rng = np.random.default_rng(RANDOM_SEED)
selected = select_ffhq_samples(ffhq_df, target_profile, rng)

print("Đã chọn xong 82 người FFHQ, khớp phân phối nhóm tuổi FG-NET:")
print(selected["age_group"].value_counts().sort_index())
print("\nPhân phối giới tính trong mẫu đã chọn (không ép buộc, tự nhiên theo pool):")
print(selected["gender"].value_counts())

## 5. Chọn ảnh distractor cho gallery FAISS

Ảnh còn lại (không trùng 82 người đã chọn làm truy vấn) dùng làm nhiễu trong gallery
— khớp đúng cấu trúc bài toán thật (database lớn hơn nhiều số ca đang tìm), tránh bài
toán tìm kiếm bị "dễ giả tạo" khi chỉ tìm trong đúng 82 ứng viên.

In [ ]:
remaining = ffhq_df[~ffhq_df["image_number"].isin(selected["image_number"])]
distractors = remaining.sample(n=min(N_DISTRACTORS, len(remaining)), random_state=RANDOM_SEED)
print(f"Đã chọn {len(distractors)} ảnh distractor cho gallery FAISS.")

## 6. Lưu kết quả

In [ ]:
selected.to_csv("ffhq_selected_82_persons.csv", index=False)
distractors.to_csv("ffhq_distractors.csv", index=False)
pd.Series(gap_distribution, name="gap").to_csv("fgnet_gap_distribution_used.csv", index=False)

print(f"✅ Đã lưu: ffhq_selected_82_persons.csv ({len(selected)} người)")
print(f"✅ Đã lưu: ffhq_distractors.csv ({len(distractors)} ảnh nhiễu cho FAISS)")
print(f"✅ Đã lưu: fgnet_gap_distribution_used.csv (để sample gap khi tạo cặp)")
print(f"\nSeed cố định: {RANDOM_SEED} — chạy lại notebook này sẽ ra đúng cùng kết quả.")

selected.head(10)

## 7. [BƯỚC TIẾP THEO — làm ở notebook pipeline chính, không phải ở đây]

1. Với mỗi ảnh trong `ffhq_selected_82_persons.csv`, chạy MiVOLO trên ảnh thật để có
   `source_age` cụ thể (không chỉ `age_group` dạng khoảng) — cần GPU, chạy trong
   notebook pipeline chính.
2. Với mỗi người, sample 2 giá trị gap từ `fgnet_gap_distribution_used.csv`, tính
   `target_age = source_age_mivolo + gap`.
3. Chạy pipeline FADING (align → inversion → editing) như đã làm cho FG-NET.
4. `id_score = cosine(generated, source_img_gốc)` — **không** so với ảnh thứ 3 nào
   khác, vì FFHQ không có ảnh thật ở tuổi đích (khác FG-NET).